# Rotating-mode spectrogram — DIII-D shot 174446

Demonstrates the MODESPEC-style pipeline from `magnetics.core.spectral`:

> **load → downsample → (optionally integrate) → compute spectrogram → de-noise → plot**

Two LFS-midplane Mirnov probes (`mpi66m307d` at φ=307°, `mpi66m340d` at φ=340°)
measure d*B*/d*t* at 200 kHz. Their toroidal separation Δφ = −33° lets us assign a
toroidal mode number *n* to each (time, frequency) cell from the cross-phase.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from magnetics.core.spectral import (
    compute_spectrogram,
    denoise_spectrogram,
    downsample,
)

# Point this at the hackathon_data directory containing magnetics_174446/
DATA_DIR = Path.home() / "hackathon_data"

## 1. Load the two probes

In [ ]:
def load_probe(data_dir: Path, angle: int) -> dict:
    path = data_dir / f"magnetics_174446/174446_mpi66m{angle}d.npz"
    d = np.load(path, allow_pickle=True)
    return {
        "time_s": d["time_ms"].astype(np.float64) * 1e-3,
        "signal": d["signal"].astype(np.float64),
        "phi": float(angle),
        "fs": float(d["fs_hz"]),
    }


p1 = load_probe(DATA_DIR, 307)
p2 = load_probe(DATA_DIR, 340)
delta_phi = p1["phi"] - p2["phi"]  # -33 degrees

print(f"probe 1: φ={p1['phi']:.0f}°, {len(p1['signal'])} samples @ {p1['fs']*1e-3:.0f} kHz")
print(f"probe 2: φ={p2['phi']:.0f}°, {len(p2['signal'])} samples @ {p2['fs']*1e-3:.0f} kHz")
print(f"Δφ = {delta_phi:.0f}°")

## 2. Downsample to the analysis window

Trim to 1500–3000 ms (where MHD activity is expected) and resample to 50 kHz —
enough to resolve modes up to 25 kHz.

In [ ]:
t_range = (1.5, 3.0)  # seconds
t1, s1 = downsample(p1["time_s"], p1["signal"], t_range=t_range, sample_rate=50_000)
t2, s2 = downsample(p2["time_s"], p2["signal"], t_range=t_range, sample_rate=50_000)

print(f"downsampled to {len(s1)} samples over [{t1[0]*1e3:.0f}, {t1[-1]*1e3:.0f}] ms")

## 3. Compute the spectrogram

A sliding 8 ms FFT window with 50% overlap. Each cell carries power, coherence,
and the cross-phase mode number *n*.

In [ ]:
spec = compute_spectrogram(t1, s1, s2, delta_phi, slice_duration=0.008)
print(f"spectrogram: {spec.power.shape[0]} time slices × {spec.power.shape[1]} frequencies")
print(f"mode-number range resolved: {spec.mode_indices.min()} … {spec.mode_indices.max()}")

## 4. De-noise

`denoise_spectrogram` removes low-amplitude / incoherent cells two ways:

1. **Coherence gate** — drop cells where the two probes are incoherent (no real mode).
2. **Per-frequency power floor** — drop cells below `k × median(power)` at each frequency.

Gated cells get power 0 and `rms_by_mode` is recomputed from what survives. Note the
floor assumes modes are *transient*; a perfectly persistent mode raises its own
per-frequency median, so lean on the coherence gate for steady/locked modes.

In [ ]:
clean = denoise_spectrogram(spec, coherence_min=0.5, power_floor_k=3.0)

raw_nz = np.count_nonzero(spec.power)
clean_nz = np.count_nonzero(clean.power)
print(f"cells kept: {clean_nz}/{raw_nz} ({100*clean_nz/raw_nz:.0f}%)")
print(f"power retained: {100*clean.power.sum()/spec.power.sum():.0f}%")

## 5. Plot — raw vs. de-noised

Left column raw, right column de-noised. Top row log power, bottom row toroidal
mode number (masked to cells with surviving power).

In [ ]:
def power_panel(ax, sp, title):
    pc = ax.pcolormesh(
        sp.time * 1e3,
        sp.frequency * 1e-3,
        np.log10(sp.power.T + 1e-20),
        shading="auto",
        cmap="inferno",
    )
    fig.colorbar(pc, ax=ax, label="log₁₀ power")
    ax.set_ylabel("Frequency (kHz)")
    ax.set_title(title)
    ax.set_ylim(0, 25)


def mode_panel(ax, sp, title, *, mask):
    mode_plot = np.where(mask.T, sp.mode_number.T.astype(float), np.nan)
    pc = ax.pcolormesh(
        sp.time * 1e3,
        sp.frequency * 1e-3,
        mode_plot,
        shading="auto",
        cmap="RdBu_r",
        vmin=-4,
        vmax=4,
    )
    fig.colorbar(pc, ax=ax, label="toroidal mode number n")
    ax.set_ylabel("Frequency (kHz)")
    ax.set_xlabel("Time (ms)")
    ax.set_title(title)
    ax.set_ylim(0, 25)


fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True, sharey=True)

power_panel(axes[0, 0], spec, "Raw — power")
power_panel(axes[0, 1], clean, "De-noised — power")

# raw mode panel: gate on coherence; de-noised: gate on surviving power
mode_panel(axes[1, 0], spec, "Raw — n (coherence > 0.5)", mask=spec.coherence > 0.5)
mode_panel(axes[1, 1], clean, "De-noised — n", mask=clean.power > 0)

fig.suptitle("DIII-D shot 174446 — rotating-mode spectrogram", fontsize=14)
plt.tight_layout()
plt.show()